# Mapping and Visualization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/05-mapping-visualization.ipynb)

This notebook teaches you how to create maps and visualizations:

- Creating choropleth maps
- Export formats (PNG, PDF, GeoJSON)
- Customization options
- Interactive maps with Folium
- Complete visualization workflows

## Setup

In [ ]:
!pip install -q socialmapper[routing] folium

In [ ]:
import os
os.environ["SOCIALMAPPER_DEMO_MODE"] = "true"

from socialmapper import (
    create_isochrone,
    get_census_blocks,
    get_census_data,
    create_map
)
print("Ready!")

## What is a Choropleth Map?

A **choropleth map** colors geographic areas according to a data variable. Darker/more intense colors typically represent higher values.

Common uses:
- Population density
- Income levels
- Election results
- Health statistics

## Basic Map Creation

In [ ]:
# Step 1: Get geographic areas
blocks = get_census_blocks(
    location=(47.6062, -122.3321),  # Seattle
    radius_km=3
)
print(f"Found {len(blocks)} census block groups")

# Step 2: Get data for those areas
geoids = [b['geoid'] for b in blocks]
census_result = get_census_data(geoids, variables=["population"])

# Step 3: Combine geometry with data
for block in blocks:
    data = census_result.data.get(block['geoid'], {})
    block['population'] = data.get('population', 0) or 0

# Step 4: Create the map
map_result = create_map(
    data=blocks,
    column="population",
    title="Population by Census Block Group"
)

print(f"\nMap created!")
print(f"Format: {map_result.format}")
print(f"Size: {len(map_result.image_data)} bytes")

## Viewing Maps in Colab

In [ ]:
from IPython.display import Image, display

# Create and display a map
map_result = create_map(
    data=blocks,
    column="population",
    title="Seattle Population"
)

# Display in notebook
display(Image(map_result.image_data))

## Export Formats

In [ ]:
# PNG (default)
png_result = create_map(data=blocks, column="population", export_format="png")
with open("population_map.png", "wb") as f:
    f.write(png_result.image_data)
print("Saved: population_map.png")

# PDF
pdf_result = create_map(data=blocks, column="population", export_format="pdf")
with open("population_map.pdf", "wb") as f:
    f.write(pdf_result.image_data)
print("Saved: population_map.pdf")

# GeoJSON (for web mapping)
geojson_result = create_map(data=blocks, column="population", export_format="geojson")
print(f"\nGeoJSON features: {len(geojson_result.geojson_data['features'])}")

## Saving Maps Directly

In [ ]:
# Save directly during creation
result = create_map(
    data=blocks,
    column="population",
    title="Seattle Population by Block Group",
    save_path="seattle_population.png"
)

print(f"Saved to: {result.file_path}")

# Display
display(Image(filename="seattle_population.png"))

## Population Density Map

In [ ]:
# Create isochrone and get blocks
isochrone = create_isochrone("Boston, MA", travel_time=20)
blocks = get_census_blocks(polygon=isochrone)
print(f"Found {len(blocks)} census block groups")

# Get population data
geoids = [b['geoid'] for b in blocks]
census_result = get_census_data(geoids, variables=["population"])

# Calculate population density
for block in blocks:
    data = census_result.data.get(block['geoid'], {})
    pop = data.get('population', 0) or 0
    area = block['area_sq_km'] if block['area_sq_km'] > 0 else 0.01
    
    block['population'] = pop
    block['density'] = pop / area  # People per km²

# Create density map
result = create_map(
    data=blocks,
    column="density",
    title="Population Density (people/km²)",
    save_path="boston_density.png"
)

print(f"Map saved to: {result.file_path}")
display(Image(filename="boston_density.png"))

## Income Map

In [ ]:
# Get income data
isochrone = create_isochrone("San Francisco, CA", travel_time=15)
blocks = get_census_blocks(polygon=isochrone)

geoids = [b['geoid'] for b in blocks]
census_result = get_census_data(geoids, variables=["median_income"])

# Add income to blocks
for block in blocks:
    data = census_result.data.get(block['geoid'], {})
    block['median_income'] = data.get('median_income', 0) or 0

# Filter out blocks with no income data
blocks_with_data = [b for b in blocks if b['median_income'] > 0]
print(f"Blocks with income data: {len(blocks_with_data)}")

# Create map
result = create_map(
    data=blocks_with_data,
    column="median_income",
    title="Median Household Income by Block Group",
    save_path="sf_income.png"
)

display(Image(filename="sf_income.png"))

## Multiple Variable Maps

In [ ]:
location = "Chicago, IL"
isochrone = create_isochrone(location, travel_time=20)
blocks = get_census_blocks(polygon=isochrone)

# Get multiple variables
geoids = [b['geoid'] for b in blocks]
census_result = get_census_data(
    geoids,
    variables=["population", "median_income", "median_age"]
)

# Add all variables to blocks
for block in blocks:
    data = census_result.data.get(block['geoid'], {})
    block['population'] = data.get('population', 0) or 0
    block['median_income'] = data.get('median_income', 0) or 0
    block['median_age'] = data.get('median_age', 0) or 0

# Create maps for each variable
variables = [
    ('population', 'Population'),
    ('median_income', 'Median Income'),
    ('median_age', 'Median Age')
]

for var, title in variables:
    # Filter blocks with data
    valid_blocks = [b for b in blocks if b[var] > 0]
    
    if valid_blocks:
        result = create_map(
            data=valid_blocks,
            column=var,
            title=f"{title} - Chicago Area",
            save_path=f"chicago_{var}.png"
        )
        print(f"Created: chicago_{var}.png")

In [ ]:
# Display all maps
from IPython.display import HTML

display(HTML("<h3>Population</h3>"))
display(Image(filename="chicago_population.png"))

display(HTML("<h3>Median Income</h3>"))
display(Image(filename="chicago_median_income.png"))

display(HTML("<h3>Median Age</h3>"))
display(Image(filename="chicago_median_age.png"))

## Interactive Maps with Folium

In [ ]:
import folium
from shapely.geometry import shape

# Get data
location = (47.6062, -122.3321)  # Seattle
blocks = get_census_blocks(location=location, radius_km=2)
geoids = [b['geoid'] for b in blocks]
census_result = get_census_data(geoids, variables=["population"])

# Add population to blocks
for block in blocks:
    data = census_result.data.get(block['geoid'], {})
    block['population'] = data.get('population', 0) or 0

# Create Folium map
m = folium.Map(location=list(location), zoom_start=13)

# Add each block as a polygon
max_pop = max(b['population'] for b in blocks) or 1

for block in blocks:
    pop = block['population']
    
    # Color scale based on population
    intensity = pop / max_pop
    if intensity > 0.7:
        color = '#d73027'  # Dark red
    elif intensity > 0.4:
        color = '#fc8d59'  # Orange
    elif intensity > 0.2:
        color = '#fee08b'  # Yellow
    else:
        color = '#91cf60'  # Green
    
    folium.GeoJson(
        block['geometry'],
        style_function=lambda x, c=color: {
            'fillColor': c,
            'color': 'black',
            'weight': 1,
            'fillOpacity': 0.6
        },
        tooltip=f"Population: {pop:,}"
    ).add_to(m)

# Display map
m

## Adding Markers to Folium Maps

In [ ]:
from socialmapper import get_poi

# Get hospitals
hospitals = get_poi(location, categories=["hospital"], limit=10)

# Add hospital markers to the map
for h in hospitals:
    folium.Marker(
        location=[h['lat'], h['lon']],
        popup=h['name'],
        icon=folium.Icon(color='red', icon='plus')
    ).add_to(m)

print(f"Added {len(hospitals)} hospital markers")
m

## Exporting GeoJSON for Web Maps

In [ ]:
import json

# Create map and export as GeoJSON
result = create_map(
    data=blocks,
    column="population",
    export_format="geojson"
)

# Save for web
with open("seattle_blocks.geojson", "w") as f:
    json.dump(result.geojson_data, f, indent=2)

print("Saved: seattle_blocks.geojson")
print(f"Features: {len(result.geojson_data['features'])}")
print("\nThis file can be used with:")
print("  - Leaflet")
print("  - Mapbox")
print("  - QGIS")
print("  - Any GeoJSON-compatible tool")

## Map Metadata

In [ ]:
result = create_map(
    data=blocks,
    column="population",
    title="Population Map"
)

# Access metadata
print("Map Metadata:")
for key, value in result.metadata.items():
    print(f"  {key}: {value}")

## Complete Visualization Workflow

In [ ]:
def create_demographic_map(location, travel_time=15, variable="population"):
    """Create a complete demographic map for a location."""
    
    print(f"Creating {variable} map for {location}...")
    
    # Get area of interest
    isochrone = create_isochrone(location, travel_time=travel_time)
    
    # Get census blocks
    blocks = get_census_blocks(polygon=isochrone)
    print(f"  Block groups: {len(blocks)}")
    
    # Get census data
    geoids = [b['geoid'] for b in blocks]
    census_result = get_census_data(geoids, variables=[variable])
    
    # Combine
    for block in blocks:
        data = census_result.data.get(block['geoid'], {})
        block[variable] = data.get(variable, 0) or 0
    
    # Filter valid data
    valid_blocks = [b for b in blocks if b[variable] > 0]
    print(f"  Blocks with data: {len(valid_blocks)}")
    
    # Create map
    filename = f"{location.replace(', ', '_').replace(' ', '_').lower()}_{variable}.png"
    
    result = create_map(
        data=valid_blocks,
        column=variable,
        title=f"{variable.replace('_', ' ').title()} - {location}",
        save_path=filename
    )
    
    print(f"  Saved: {filename}")
    return result, valid_blocks

# Create multiple maps
for var in ["population", "median_income"]:
    result, blocks = create_demographic_map("Portland, OR", variable=var)
    display(Image(filename=result.file_path))

## Best Practices

In [ ]:
# 1. Always filter out missing data
valid_blocks = [b for b in blocks if b.get('population', 0) > 0]

# 2. Handle zero values for density calculations
for block in blocks:
    area = block['area_sq_km'] if block['area_sq_km'] > 0 else 0.01
    block['density'] = block.get('population', 0) / area

# 3. Verify column exists before mapping
column = 'population'
if column in blocks[0]:
    result = create_map(data=blocks, column=column)
else:
    print(f"Column '{column}' not found")

# 4. Use descriptive titles
result = create_map(
    data=blocks,
    column="density",
    title="Population Density (people/km²) - Seattle Metro"
)

print("Best practices applied!")

## Next Steps

Continue with:

- **[Complete Workflow](06-complete-workflow.ipynb)** - Full analysis from start to finish
- **[Food Desert Case Study](07-food-desert-case-study.ipynb)** - Real-world mapping application